# 02. Baselines

Milestone 2: the forecasts every later model has to beat. For VIX (and, in the
command, the other implied-vol indices), at horizons 1, 5 and 21 sessions:

* `rw`: the random walk, tomorrow's VIX is today's;
* `hist_mean`, `hist_mean_252`: the mean of the levels up to the cutoff (expanding, or the last 252 sessions);
* `arima_log`: an ARMA(p, q) with a constant on ln VIX (d = 0), (p, q) by AIC, refitted at every cutoff;
* `har_log`: Corsi's HAR on ln VIX (daily, weekly, monthly averages), one OLS per horizon.

The walk-forward protocol is ADR 0001 D4: folds of 63 sessions anchored on
`holdout_start`, parameters frozen inside a fold, the inputs moving daily, and
**no row on or after `holdout_start` is read** (`walkforward.pre_holdout_inputs`).
Only dev rows whose target is dated before `holdout_start` are scored.

The full run is the command (about 4 minutes on real data, most of it ARIMA):

```
uv run python -m ivf.models baselines            # real data, local only
uv run python -m ivf.models --synthetic baselines
```

This notebook runs VIX on the last `IVF_NOTEBOOK_FOLDS` dev folds (default 4) so it
stays quick. **Licensing:** real-data numbers stay on your machine (ADR 0001 D2).

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import json
import math
import os
import warnings

import numpy as np
import pandas as pd

from ivf import config, protocol
from ivf.data import build_interim, calendar, check, synthetic
from ivf.evaluation import walkforward
from ivf.features import build
from ivf.models import baselines, registry

warnings.filterwarnings("ignore", category=FutureWarning)


def load():
    """(mode, protocol, iv, price, build_report): real if available, else synthetic."""
    if os.environ.get("IVF_NOTEBOOK_DATA") != "synthetic":
        try:
            proto = protocol.load_protocol(protocol.protocol_path())
            proto.require_data_end()
            iv, price, report = build_interim.read_interim(config.data_layout().root)
            return "real", proto, iv, price, report
        except (protocol.ProtocolError, build_interim.InterimError) as why:
            print("no real data:", why)
    proto = protocol.load_protocol(protocol.protocol_path(synthetic=True))
    iv, price = synthetic.synthetic_market(proto)
    report = {"files": [{"symbol": s, "fetched_at": "2026-10-27T12:00:00.000000Z", "duplicates_dropped": 0, "duplicate_dates": []}
                        for s in list(iv["symbol"].unique()) + ["^GSPC"]]}
    return "synthetic", proto, iv, price, report


MODE, PROTO, IV, PRICE, REPORT = load()
RESULT = check.frozen_check(PROTO, IV, PRICE, REPORT)
if not RESULT.ok:
    raise RuntimeError("check reports VIOLATED: run `uv run python -m ivf.data check` and fix the data first")
print("data:", MODE.upper(), "| holdout_start", PROTO.holdout_start, "| data_end", PROTO.data_end)

## 1. Folds, and what the run may read

The fold schedule is computed from the calendar alone (no data), once, and passed
into the run. The inputs are M1's `eda_inputs`: sessions and rows strictly before
`holdout_start`.

In [ ]:
CAL = calendar.master_calendar(PROTO)
CALENDAR_DATES = list(CAL["date"])
FOLDS = walkforward.fold_schedule(CALENDAR_DATES, PROTO.dev_start, PROTO.holdout_start, PROTO.fold_len, PROTO.data_end)
DEV = [f for f in FOLDS if f.period == "dev"]
DATES, _, IV_PRE, _ = walkforward.pre_holdout_inputs(CAL, IV, PRICE, PROTO.holdout_start)
Y = build.align(DATES, IV_PRE, "symbol", "VIX")
N_FOLDS = int(os.environ.get("IVF_NOTEBOOK_FOLDS", "4"))
print(f"{len(DEV)} dev folds; first cutoff {CALENDAR_DATES[DEV[0].cutoff_idx]}; last dev origin {CALENDAR_DATES[DEV[-1].last_origin_idx]}")
print(f"{len(DATES)} sessions readable, the last on {DATES[-1]} (before holdout_start {PROTO.holdout_start})")
print(f"this notebook runs the last {N_FOLDS} dev folds")

## 2. Fit and forecast

At each cutoff every model is fitted on the closes dated up to the cutoff, then
forecasts every origin of the fold. ARIMA records the order it chose and every
candidate's AIC; a fit that fails would keep the previous fold's parameters.

In [ ]:
DEFS, SETTINGS = registry.load_models()
MODELS = baselines.build_models(SETTINGS)
RUN = walkforward.run_symbol("VIX", "vix_level", Y, DATES, CALENDAR_DATES, DEV[-N_FOLDS:], MODELS, PROTO.horizons,
                             PROTO.holdout_start, SETTINGS.min_train_sessions)
FC = RUN.forecasts
arima = RUN.fits[RUN.fits.model_id == "arima_log"]
for _, row in arima.iterrows():
    params, hyper = json.loads(row.params_json), json.loads(row.hyperparams_json)
    print(f"cutoff {row.cutoff_date}: ARIMA{tuple(params['order'])}, d_rule {hyper['d_rule']}, "
          f"AIC {row.aic:.1f}, fallback {row.fallback}")
print(len(FC), "forecast rows;", {m: round(s, 1) for m, s in RUN.seconds.items()}, "seconds")

## 3. Losses, and which back-transform each one rewards

`yhat` is the lognormal mean `exp(mu + s2/2)`, the forecast that is evaluated;
`yhat_naive` is `exp(mu)`, the median. RMSE is consistent for the mean, MAE for
the median, MAPE for something below the median, and QLIKE on `(yhat/100)^2`
for the mean of the variance, which is above the mean of y. So the `median` rows
can win on MAE and MAPE without being better forecasts of the mean
(`ivf/evaluation/losses.py`, ADR 0001 4.3).

In [ ]:
METRIC = walkforward.metric_table(FC, PROTO.holdout_start, PROTO.document["min_cell_n"], {"arima_log", "har_log"})
pd.set_option("display.width", 140)
print(METRIC.pivot_table(index=["horizon", "model_id", "forecast"], values=["rmse", "mae", "mape", "qlike", "n"]).round(4))

## 4. Synthetic only: the true conditional mean as a floor

On synthetic data ln VIX is an AR(1) with known parameters, so the best possible
forecast is known: `x_{t+h}` given `x_t` is normal with mean
`mu + phi^h (x_t - mu)` and variance `sigma^2 (1 - phi^(2h)) / (1 - phi^2)`. Its
RMSE is the floor no model can beat on average.

In [ ]:
if MODE == "synthetic":
    p = synthetic.SyntheticParams()
    rows = walkforward.scored(FC[FC.model_id == "rw"], PROTO.holdout_start)
    for h in PROTO.horizons:
        r = rows[rows.horizon == h]
        x_t = np.log(r["yhat"].to_numpy())  # the random walk's forecast is y_t
        m = p.mu + p.phi**h * (x_t - p.mu)
        s2 = p.sigma**2 * (1 - p.phi ** (2 * h)) / (1 - p.phi**2)
        truth = np.exp(m + s2 / 2)
        print(f"h={h:2}: RMSE of the true conditional mean {math.sqrt(np.mean((r['y'] - truth) ** 2)):.4f} (n = {len(r)})")
else:
    print("real data: no known truth")

In [ ]:
import matplotlib.pyplot as plt

last = FC[(FC.horizon == 21) & (FC.fold_id == DEV[-1].fold_id)]
fig, ax = plt.subplots(figsize=(9, 3.5))
for model_id, g in last.groupby("model_id"):
    ax.plot(g["target_date"], g["yhat"], lw=1, label=model_id)
actual = last[last.model_id == "rw"].dropna(subset=["y"])
ax.plot(actual["target_date"], actual["y"], "k.", ms=3, label="actual")
ax.set_title(f"VIX 21 sessions ahead, last dev fold ({MODE} data); targets in the holdout are not shown")
ax.legend(fontsize=7, ncol=3)
fig.tight_layout()  # no plt.show(): Jupyter draws the figure at the end of the cell